# Private Markets PDF Extraction Walkthrough

This notebook turns the project outputs into a short analytical walkthrough. It focuses on what the pipeline actually extracted, which source worked best, and where analyst review is still needed.

## Problem framing

Private-markets data often arrives as PDF reports rather than structured tables. Even when the documents are public, analysts still need to:

- locate the right source pages
- extract rows from inconsistent layouts
- normalize field names and datatypes
- preserve provenance for auditability
- review suspicious rows instead of trusting every parse

This project uses a small public CalPERS source set to reproduce that workflow.

In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'

records = pd.read_csv(DATA_DIR / 'normalized' / 'fund_records_validated.csv')
audit = pd.read_csv(OUTPUTS_DIR / 'samples' / 'source_quality_audit.csv')
sample = pd.read_csv(OUTPUTS_DIR / 'samples' / 'high_confidence_sample.csv')
issues = pd.read_csv(OUTPUTS_DIR / 'logs' / 'validation_issues.csv')
summary = json.loads((DATA_DIR / 'normalized' / 'dashboard_summary.json').read_text())

summary

## Source inventory and quality audit

The core question is not just how many rows were extracted, but which source produced the cleanest fund-level data.

In [ ]:
audit

## Walkthrough 1: strongest source

The strongest source is the 2023-24 Annual Investment Report. Its private-equity holdings section uses a stable line format:

`Security Name | Book Value | Market Value`

That makes it feasible to parse high-confidence fund rows and map `Market Value` conservatively to `nav`.

In [ ]:
annual_text = json.loads((DATA_DIR / 'extracted' / 'annual-investment-report-fy-2024_text.json').read_text())
page_310 = next(page for page in annual_text['pages'] if page['page_number'] == 310)
page_310['lines'][:12]

In [ ]:
records[records['source_file'] == 'annual-investment-report-fy-2024.pdf'][['page_number', 'fund_name', 'nav', 'confidence_flag', 'notes']].head(20)

## Walkthrough 2: harder source

The annual program review PDFs are a good example of why private-markets PDF extraction is difficult. They contain useful information, but much of it appears in chart fragments, strategy mix summaries, and multi-line commentary rather than clean fund tables.

In [ ]:
review_text = json.loads((DATA_DIR / 'extracted' / '202506-invest-agenda-item06c-01_text.json').read_text())
page_10 = next(page for page in review_text['pages'] if page['page_number'] == 10)
page_10['lines'][:18]

In [ ]:
records[records['source_file'] == '202506-invest-agenda-item06c-01.pdf'][['page_number', 'fund_name', 'nav', 'tvpi', 'irr', 'confidence_flag', 'notes']].head(20)

## Representative normalized output

These are the highest-confidence rows chosen as representative README-ready examples.

In [ ]:
sample

## Validation findings

The pipeline deliberately keeps validation separate from extraction. Suspicious rows are not hidden; they are logged for review.

In [ ]:
issues

In [ ]:
records.groupby(['source_file', 'confidence_flag']).size().unstack(fill_value=0)

## Quick visuals

These simple views are enough to communicate the project story: the annual report provides the cleanest fund-level extraction, while the program review sources remain useful but noisy.

In [ ]:
records['source_file'].value_counts().plot(kind='bar', title='Records by source', figsize=(8, 4))

In [ ]:
records[records['confidence_flag'] == 'high']['nav'].dropna().div(1_000_000_000).hist(bins=20, figsize=(8, 4))

## Lessons learned

- The strongest source in a multi-PDF workflow is often the one with the most stable repeated layout, not the most private-markets-specific title.
- Provenance and confidence flags matter more than maximizing raw row count.
- The annual program review PDFs still need source-specific parsing, which is realistic and valuable to show.
- This project is already a credible demonstration of private-markets data normalization under messy reporting conditions.